In [ ]:
# 导入类型定义和LangGraph的START、END特殊节点
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

In [ ]:
# 定义代理状态，包含两组运算的输入和结果
class AgentState(TypedDict):
    number1: int        # 第一组运算的第一个数
    operation: str      # 第一组运算符
    number2: int        # 第一组运算的第二个数
    finalNumber: int    # 第一组运算结果
    number3: int        # 第二组运算的第一个数
    operation2: str     # 第二组运算符
    number4: int        # 第二组运算的第二个数
    finalNumber2: int   # 第二组运算结果

In [ ]:
# 第一组加法节点
def adder(state:AgentState) -> AgentState:
    """This node adds the 2 numbers"""
    print("adder")
    state["finalNumber"] = state["number1"] + state["number2"]
    return state

# 第一组减法节点
def subtractor(state:AgentState) -> AgentState:
    """This node subtracts the 2 numbers"""
    print("subtractor")
    state["finalNumber"] = state["number1"] - state["number2"]
    return state

# 第一组路由函数：根据运算符决定走加法还是减法
def decide_next_node(state:AgentState) -> AgentState:
    """This node will select the next phase"""
    if state["operation"] == "+":
        return "addition_operation"
    
    elif state["operation"] == "-":
        print("SUB 1")
        return "subtraction_operation"   

# 第二组加法节点
def adder2(state:AgentState) -> AgentState:
    """This node adds the 2 numbers"""
    print("adder1")
    state["finalNumber2"] = state["number3"] + state["number4"]
    print(state["finalNumber2"])

    return state

# 第二组减法节点
def subtractor2(state:AgentState) -> AgentState:
    """This node subtracts the 2 numbers"""
    print("subtractor1")
    state["finalNumber2"] = state["number3"] - state["number4"]
    print(state["finalNumber2"])
    return state

# 第二组路由函数：根据运算符决定走加法还是减法
def decide_next_node1(state:AgentState) -> AgentState:
    """This node will select the next phase"""
    if state["operation2"] == "+":
        print("ADD1")
        return "addition_operation2"
    
    elif state["operation2"] == "-":
        return "subtraction_operation2"   

In [ ]:
# 构建包含两阶段条件路由的状态图
graph = StateGraph(AgentState)

# 添加第一组运算节点
graph.add_node("add_node", adder)
graph.add_node("subtract_node", subtractor)
graph.add_node("router", lambda state:state)  # 第一个路由器（透传节点）

# 添加第二组运算节点
graph.add_node("add_node2", adder2)
graph.add_node("subtract_node2", subtractor2)
graph.add_node("router2", lambda state:state)  # 第二个路由器（透传节点）

# 从起始节点进入第一个路由器
graph.add_edge(START, "router")

# 第一阶段条件路由：根据operation选择加法或减法
graph.add_conditional_edges(
    "router", 
    decide_next_node,
    {
        # Edge: Node format
        "addition_operation": "add_node",
        "subtraction_operation": "subtract_node"
    }
)

# 第一阶段运算完成后进入第二个路由器
graph.add_edge("add_node", "router2")
graph.add_edge("subtract_node", "router2")

# 第二阶段条件路由：根据operation2选择加法或减法
graph.add_conditional_edges(
    "router2", 
    decide_next_node1,
    {
        # Edge: Node format
        "addition_operation2": "add_node2",
        "subtraction_operation2": "subtract_node2"
    }
)

# 第二阶段运算完成后结束
graph.add_edge("add_node2", END)
graph.add_edge("subtract_node2", END)

# 编译图
app = graph.compile()

In [ ]:
# 可视化图结构
from IPython.display import Image, display
display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
# 创建初始状态：第一组做减法(10-5)，第二组做加法(7+2)
initial_state = AgentState(number1 = 10, operation="-", number2 = 5, number3 = 7, number4=2, operation2="+", finalNumber= 0, finalNumber2 = 0)

In [ ]:
# 执行图并打印最终结果
print(app.invoke(initial_state))